In [1]:
import collections
from glob import glob
import joblib
import pandas as pd
from matplotlib import pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import os
import numpy as np
import torch.nn as nn
import torch.utils.data
from torchvision import transforms
from PIL import ImageFile
import torchvision.models as models
from PIL import Image, ImageFile
import scanpy as sc

ImageFile.LOAD_TRUNCATED_IMAGES = True


class PTC_cell_2x(torch.utils.data.Dataset):
    def __init__(self, root, slide='11_breast_cancer3',transform=None, stain_norm=False):
        super(PTC_cell_2x, self).__init__()
        self.root = root
        self.slide = slide
        self.transform = transform
        self.stain_norm = stain_norm

        patch_path = os.path.join(root, slide, '2x_patches')
        patch = os.listdir(patch_path)
        patch_list = [x.split('.j')[0] for x in patch]

        cell_label = pd.read_csv(os.path.join(root, slide, 'cell_ratio.csv'), index_col=0)
        # cell_label = cell_label.apply(lambda x: x*100.0)
        
        gene_label = pd.read_csv(os.path.join(root, slide, 'high_250_stdata.csv'), index_col=0)
        label_df = pd.merge(gene_label, cell_label, left_index=True, right_index=True)

        # label_index_set = set(label_df.index)
        # patch_index_set = set(patch_list)
        # and_set = label_index_set & patch_index_set
        # patch_list = list(and_set)
        # self.label_df = label_df.loc[patch_list]
        
        self.label_df = label_df
        self.patch = patch_list


    def __getitem__(self, index):
        patch_id = self.patch[index]
        patch_path = os.path.join(self.root, self.slide, '2x_patches', patch_id)
        patch = Image.open(patch_path+'.jpg').convert('RGB')
        data = transforms.Resize((224, 224))(patch)
        if self.transform is not None:
            data = self.transform(data)

        if patch_id in self.label_df.index:
            label = self.label_df.loc[patch_id].values
        else:
            label = np.full((self.label_df.iloc[0].values.shape), -1.0)
        label = torch.Tensor(label)

        return patch_id, data, label

    def __len__(self):
        return len(self.patch)


class fully_connected(nn.Module):
    def __init__(self, model, num_ftrs, num_classes):
        super(fully_connected, self).__init__()
        self.model = model
        self.fc_4 = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        x = self.model(x)
        x = torch.flatten(x, 1)
        out_1 = x
        out_3 = self.fc_4(x)
        out_3 = torch.relu(out_3)
        return out_1, out_3

In [2]:
# prepare necessary arguments and WSI sample list

tif_list = '/data1/r20user3/shared_project/Hist2Cell/data/stnet/'
tif_list = glob(tif_list + '/*[!xlsx|ipynb]')
tif_list.sort()
tif_list

['/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23209_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23268_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_C1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_C2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23269_D1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23270_E2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_D2',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_E1',
 '/data1/r20user3/shared_project/Hist2Cell/data/stnet/23272_E2',
 '/data1/r20user3/shared_

In [3]:
# define test slides list

test_slides = list()
for tif in tif_list:
    tif_path = tif.split('/')[-1]
    test_slides.append(tif_path)
test_slides

['23209_C1',
 '23209_C2',
 '23209_D1',
 '23268_C1',
 '23268_C2',
 '23268_D1',
 '23269_C1',
 '23269_C2',
 '23269_D1',
 '23270_D2',
 '23270_E1',
 '23270_E2',
 '23272_D2',
 '23272_E1',
 '23272_E2',
 '23277_D2',
 '23277_E1',
 '23277_E2',
 '23287_C1',
 '23287_C2',
 '23287_D1',
 '23288_D2',
 '23288_E1',
 '23288_E2',
 '23377_C1',
 '23377_C2',
 '23377_D1',
 '23450_D2',
 '23450_E1',
 '23450_E2',
 '23506_C1',
 '23506_C2',
 '23506_D1',
 '23508_D2',
 '23508_E1',
 '23508_E2',
 '23567_D2',
 '23567_E1',
 '23567_E2',
 '23803_D2',
 '23803_E1',
 '23803_E2',
 '23810_D2',
 '23810_E1',
 '23810_E2',
 '23895_C1',
 '23895_C2',
 '23895_D1',
 '23901_C2',
 '23901_D1',
 '23903_C1',
 '23903_C2',
 '23903_D1',
 '23944_D2',
 '23944_E1',
 '23944_E2',
 '24044_D2',
 '24044_E1',
 '24044_E2',
 '24105_C1',
 '24105_C2',
 '24105_D1',
 '24220_D2',
 '24220_E1',
 '24220_E2',
 '24223_D2',
 '24223_E1',
 '24223_E2']

In [4]:
# prepare my GPU

gpu_list = [4]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [5]:
# define test dataset transform

test_transform_pcam = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

In [10]:
batch_size = 512
patch_dir = "/data1/r20user3/shared_project/Hist2Cell/data/stnet"

graphs = dict()
for slide in test_slides:
    print(slide)
    test_data = PTC_cell_2x(root=patch_dir, slide=slide,transform=test_transform_pcam)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=4)

    spots_coord = pd.read_csv(os.path.join("/data1/r20user3/shared_project/Hist2Cell/data/stnet", slide, "2x_spots.csv"), index_col=0)

    with torch.no_grad():
        # model.eval()
        test_prediction_array = []
        test_label_array = []
        test_name_array = []

        for name, data, label in test_loader:
            test_name_array.append(list(name))
            data = data.to(device)
            label = label.to(device)
            label = label.float()
            label = label.squeeze()
            test_label_array.append(label.cpu().detach().numpy())
            
            # features, output = model(data)
            # test_prediction_array.append(features.squeeze().cpu().detach().numpy())
            test_prediction_array.append(data.cpu().detach().numpy())

    for i in range(len(test_prediction_array)):
        if len(test_prediction_array[i].shape) <= 1:
            test_prediction_array[i] = test_prediction_array[i][np.newaxis, :]
    for i in range(len(test_label_array)):
        if len(test_label_array[i].shape) <= 1:
            test_label_array[i] = test_label_array[i][np.newaxis, :]

    test_prediction_array = np.concatenate(test_prediction_array)
    test_label_array = np.concatenate(test_label_array)
    
    test_names = list()
    for names in test_name_array:
        test_names = test_names+names
    test_node_x_y = list()
    for item in test_names:
        test_node_x_y.append([float(item.split('x')[0]), float(item.split('x')[1])])

    adj = np.zeros((len(test_node_x_y), len(test_node_x_y)))

    for i in range(len(test_node_x_y)):
        for j in range(len(test_node_x_y)):
            if i == j:
                adj[i][j] = 1.0
            else:
                x1 = test_node_x_y[i][0]
                y1 = test_node_x_y[i][1]
                x2 = test_node_x_y[j][0]
                y2 = test_node_x_y[j][1]

                if x2 <= x1 - 1 or x2 >= x1 + 1 or y2 <= y1 - 1 or y2 >= y1 + 1:
                    continue
                else:
                    adj[i][j] = 1.0

    graphs[slide] = {
        'features': test_prediction_array,
        'labels': test_label_array,
        'adj': adj,
        'names': test_names,
        'coord': spots_coord.loc[test_names].values,
    }

    print("OK")

23209_C1
OK
23209_C2
OK
23209_D1
OK
23268_C1
OK
23268_C2
OK
23268_D1
OK
23269_C1
OK
23269_C2
OK
23269_D1
OK
23270_D2
OK
23270_E1
OK
23270_E2
OK
23272_D2
OK
23272_E1
OK
23272_E2
OK
23277_D2
OK
23277_E1
OK
23277_E2
OK
23287_C1
OK
23287_C2
OK
23287_D1
OK
23288_D2
OK
23288_E1
OK
23288_E2
OK
23377_C1
OK
23377_C2
OK
23377_D1
OK
23450_D2
OK
23450_E1
OK
23450_E2
OK
23506_C1
OK
23506_C2
OK
23506_D1
OK
23508_D2
OK
23508_E1
OK
23508_E2
OK
23567_D2
OK
23567_E1
OK
23567_E2
OK
23803_D2
OK
23803_E1
OK
23803_E2
OK
23810_D2
OK
23810_E1
OK
23810_E2
OK
23895_C1
OK
23895_C2
OK
23895_D1
OK
23901_C2
OK
23901_D1
OK
23903_C1
OK
23903_C2
OK
23903_D1
OK
23944_D2
OK
23944_E1
OK
23944_E2
OK
24044_D2
OK
24044_E1
OK
24044_E2
OK
24105_C1
OK
24105_C2
OK
24105_D1
OK
24220_D2
OK
24220_E1
OK
24220_E2
OK
24223_D2
OK
24223_E1
OK
24223_E2
OK


In [11]:
graphs['24223_E2']['features'].shape

(1382, 3, 224, 224)

In [12]:
graphs['24223_E2']['coord'].shape

(1382, 2)

In [13]:
graphs['24223_E2']['names']

['23.5x29.5',
 '23x18',
 '12.5x19.5',
 '22.5x14.5',
 '24x19',
 '24x31',
 '26.5x24.5',
 '25.5x26',
 '9.5x28.5',
 '16x15.5',
 '24x28',
 '21x26',
 '18x17',
 '13x26.5',
 '11.5x20',
 '22x21.5',
 '26x26.5',
 '20x22.5',
 '8.5x28',
 '7.5x24.5',
 '23.5x24.5',
 '8.5x23.5',
 '20x22',
 '20.5x23',
 '22.5x17.5',
 '18x19.5',
 '20x18.5',
 '8.5x29.5',
 '10.5x18.5',
 '9.5x30',
 '6.5x18',
 '10.5x18',
 '10.5x21',
 '12.5x20.5',
 '24x27',
 '24x30',
 '24x22.5',
 '9.5x23.5',
 '14.5x24',
 '8x17',
 '12.5x30.5',
 '25x22',
 '7x22',
 '6x20.5',
 '17x20.5',
 '17.5x20',
 '7x15.5',
 '16.5x21.5',
 '6x20',
 '15x24.5',
 '26x20.5',
 '9.5x15',
 '17x15.5',
 '15.5x24.5',
 '5.5x21',
 '21x19.5',
 '20.5x28.5',
 '19x22.5',
 '18.5x23.5',
 '12x18.5',
 '5x22.5',
 '20.5x24',
 '22x17.5',
 '24x21.5',
 '6x18',
 '22x27',
 '7x29.5',
 '12.5x31',
 '5x22',
 '7x18.5',
 '6.5x17',
 '22.5x25.5',
 '4x15',
 '6.5x17.5',
 '14x19',
 '21x27.5',
 '16.5x15.5',
 '10x19.5',
 '5.5x21.5',
 '7x16',
 '6.5x29.5',
 '24.5x24.5',
 '8.5x31.5',
 '13.5x22.5',
 '20x

In [14]:
graph = graphs
for slide in graph:
    print(slide)

23209_C1
23209_C2
23209_D1
23268_C1
23268_C2
23268_D1
23269_C1
23269_C2
23269_D1
23270_D2
23270_E1
23270_E2
23272_D2
23272_E1
23272_E2
23277_D2
23277_E1
23277_E2
23287_C1
23287_C2
23287_D1
23288_D2
23288_E1
23288_E2
23377_C1
23377_C2
23377_D1
23450_D2
23450_E1
23450_E2
23506_C1
23506_C2
23506_D1
23508_D2
23508_E1
23508_E2
23567_D2
23567_E1
23567_E2
23803_D2
23803_E1
23803_E2
23810_D2
23810_E1
23810_E2
23895_C1
23895_C2
23895_D1
23901_C2
23901_D1
23903_C1
23903_C2
23903_D1
23944_D2
23944_E1
23944_E2
24044_D2
24044_E1
24044_E2
24105_C1
24105_C2
24105_D1
24220_D2
24220_E1
24220_E2
24223_D2
24223_E1
24223_E2


In [15]:
from torch import Tensor
import torch
import os

for slide_name in graph:
    x = Tensor(graph[slide_name]['features'])
    from torch_geometric.utils import dense_to_sparse
    adj = Tensor(graph[slide_name]['adj'])
    edge_index, _ = dense_to_sparse(adj)
    y = Tensor(graph[slide_name]['labels'])
    from torch_geometric.data import Data
    pos = Tensor(graph[slide_name]['coord'])

    data = Data(x=x, edge_index=edge_index, y=y, pos=pos)
    
    torch.save(data, os.path.join("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet_2x", slide_name+".pt"))

In [16]:
import torch

data1 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet_2x/23209_C1.pt")
data2 = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/stnet_2x/23209_C2.pt")

In [17]:
from torch_geometric.data import Batch

data = Batch.from_data_list([data1, data2])
data

DataBatch(x=[2314, 3, 224, 224], edge_index=[2, 19742], y=[2314, 289], pos=[2314, 2], batch=[2314], ptr=[3])

In [18]:
from random import shuffle
from torch_geometric.loader import NeighborLoader
import torch_geometric

torch_geometric.typing.WITH_PYG_LIB = False

from torch_geometric.seed import seed_everything
seed_everything(0)

loader = NeighborLoader(
    data,
    # Sample 30 neighbors for each node for 2 iterations
    num_neighbors=[-1, -1],
    # Use a batch size of 128 for sampling training nodes
    batch_size=1,
    directed=False,
    input_nodes=None,
    # disjoint=True,
    shuffle=True
)

i = 0
for sampled_data in loader:
    print(sampled_data.input_id)
    i = i + 1
    if i > 10:
        break

tensor([344])
tensor([1438])
tensor([1927])
tensor([1778])
tensor([1531])
tensor([1655])
tensor([761])
tensor([1223])
tensor([92])
tensor([1810])
tensor([1663])


/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "


In [19]:
sampled_data

DataBatch(x=[25, 3, 224, 224], edge_index=[2, 169], y=[25, 289], pos=[25, 2], ptr=[3], n_id=[25], e_id=[169], input_id=[1], batch_size=1)

In [20]:
sampled_data.pos

tensor([[6078.3149, 4603.7461],
        [6237.2915, 4608.0176],
        [6236.8711, 4755.7764],
        [6234.4355, 4462.0996],
        [6075.4590, 4457.8281],
        [6074.8584, 4745.3911],
        [5923.6128, 4743.4800],
        [5927.0693, 4601.8350],
        [5924.2134, 4455.9170],
        [6395.8477, 4760.0479],
        [6396.2681, 4612.2891],
        [6396.9863, 4464.9751],
        [6233.4146, 4897.4214],
        [6395.4268, 4907.8071],
        [6071.4019, 4887.0361],
        [6397.7051, 4317.6611],
        [6072.6030, 4311.9102],
        [6235.1538, 4314.7856],
        [5922.4707, 4311.4937],
        [5924.1323, 4886.7793],
        [5776.8628, 4886.5229],
        [5775.8242, 4599.9238],
        [5776.3433, 4743.2236],
        [5774.0811, 4455.5005],
        [5772.3379, 4311.0771]])